In [2]:
from pathlib import Path
import csv
import gc
import hashlib
import sys

import numpy as np
import pyarrow.parquet as pq
from tokenizers import Tokenizer
from tqdm.auto import tqdm


# ============================================================
# IMPORTS FROM PROJECT
# ============================================================

PROJECT_ROOT = Path("/home/jonas/PycharmProjects/tikzcodegenerator")
sys.path.insert(0, str(PROJECT_ROOT))

from evaluation.promptfoo.pf_utils.tikz_rendering import (
    TikzRenderError,
    render_tex_to_png,
)

from data.clean_text_in_tex import replace_all_nodes


# ============================================================
# CONFIG
# ============================================================

p_val = 0.5
p_code_corpus = 5
seed = 42

max_reference_tokens = 8192
batch_size = 512  # kleiner = weniger RAM, groesser = schneller

# Wenn True: Rows, deren TikZ-Code nicht gerendert werden kann, werden uebersprungen.
# Wenn False: Das Skript bricht beim ersten Render-Fehler ab.
skip_render_failures = False

# Ohne Text: "" entfernt Node-Inhalte komplett. Alternative: "Text" als Platzhalter.
node_replacement_without_text = "Text"

tokenizer_name = "/home/jonas/models/huggingface/hub/gemma-4-31B-it-unsloth-bnb-4bit"
tokenizer = Tokenizer.from_file(str(Path(tokenizer_name) / "tokenizer.json"))

datasets = [
    {
        "source": "datikz_v4",
        "path": Path("/home/jonas/Datasets/TikZ/parquet_files/DaTikZ-v4"),
        "description_col": "vlm_description",
        "code_col": "tikz_code",
        "sample_percent": 100.0,
    },
    {
        "source": "geotikz_bridge_base",
        "path": Path("/home/jonas/Datasets/TikZ/parquet_files/geotikz_bridge_base"),
        "description_col": None,  # keine VLM description vorhanden -> leerer String
        "code_col": "response",
        "sample_percent": 50.0,
    },
]

output_dir = Path("/home/jonas/Datasets/TikZ/train-big")

image_with_text_folder = output_dir / "images_with_text"
image_without_text_folder = output_dir / "images_without_text"
code_with_text_folder = output_dir / "references_with_text"
code_without_text_folder = output_dir / "references_without_text"
description_folder = output_dir / "vlm_descriptions"
code_corpus_folder = output_dir / "code_corpus"
render_failed_folder = output_dir / "render_failed"

manifest_columns = [
    "code_path_with_text",
    "code_path_without_text",
    "image_path_with_text",
    "image_path_without_text",
    "vlm_description_path",
    "input_image",
    "reference_image",
    "reference_code",
    "input_image_format_for_vlm",
    "vlm_description",
]


# ============================================================
# SMALL HELPERS
# ============================================================

def normalize_text(value) -> str:
    return "" if value is None else str(value)


def is_empty_code(text: str) -> bool:
    text = text.strip()
    return not text or text.lower() == "nan"


def token_count(text: str) -> int:
    return len(tokenizer.encode(str(text), add_special_tokens=False).ids)


def keep_by_dataset_sample(source: str, source_idx: int, sample_percent: float) -> bool:
    """
    Deterministische Auswahl pro Dataset.

    100.0 => alles behalten
    50.0  => ca. 50% behalten
    0.0   => nichts behalten

    Wichtig:
    Die Auswahl ist stabil zwischen Pass 1 und Pass 2,
    solange source, source_idx und seed gleich bleiben.
    """
    if sample_percent >= 100:
        return True

    if sample_percent <= 0:
        return False

    key = f"{seed}:{source}:{source_idx}".encode("utf-8")
    digest = hashlib.blake2b(key, digest_size=8).digest()
    value = int.from_bytes(digest, byteorder="big") / 2**64

    return value < (sample_percent / 100.0)


def setup_output_folders() -> None:
    folders = [
        image_with_text_folder,
        image_without_text_folder,
        code_with_text_folder,
        code_without_text_folder,
        description_folder,
        code_corpus_folder,
    ]

    if skip_render_failures:
        folders.append(render_failed_folder)

    for folder in folders:
        folder.mkdir(parents=True, exist_ok=True)


def render_or_handle_failure(tex_code: str, output_path: Path, stem: str, variant: str) -> bool:
    metrics = {}

    try:
        render_tex_to_png(tex_code, output_path, metrics=metrics)
        return True
    except TikzRenderError as exc:
        if not skip_render_failures:
            raise

        log_path = render_failed_folder / f"{stem}_{variant}.txt"
        log_path.write_text(
            "Render failed\n"
            f"stem: {stem}\n"
            f"variant: {variant}\n"
            f"output_path: {output_path}\n"
            f"metrics: {getattr(exc, 'metrics', {})}\n\n"
            f"error:\n{exc}\n",
            encoding="utf-8",
        )
        return False


def print_config() -> None:
    if p_val + p_code_corpus >= 100:
        raise ValueError("p_val + p_code_corpus muss kleiner als 100 sein.")

    print(f"Output folder: {output_dir}")
    print(f"Batch size: {batch_size}")
    print(f"Max reference tokens: {max_reference_tokens}")
    print(f"Validation split: {p_val}%")
    print(f"Code corpus split: {p_code_corpus}%")
    print(f"Skip render failures: {skip_render_failures}")
    print(f"Node replacement without text: {node_replacement_without_text!r}")

    print("\nDataset sampling:")
    for dataset in datasets:
        print(
            f"- {dataset['source']}: "
            f"{float(dataset.get('sample_percent', 100.0))}%"
        )


# ============================================================
# PASS 1: COUNT VALID ITEMS ONLY
# ============================================================

def count_valid_items() -> tuple[int, int, int, int, int]:
    valid_count = 0
    total_seen = 0
    too_long = 0
    empty_code = 0
    sampled_out = 0

    print("\nFirst pass: filtering by dataset sample and token length...")

    for dataset in datasets:
        source = dataset["source"]
        sample_percent = float(dataset.get("sample_percent", 100.0))
        source_idx = 0
        parquet_files = sorted(dataset["path"].glob("*.parquet"))

        print(f"\nSource: {source}")
        print(f"Parquet files: {len(parquet_files)}")
        print(f"Sample percent: {sample_percent}%")

        for parquet_file in tqdm(parquet_files, desc=f"Filtering {source}"):
            pf = pq.ParquetFile(parquet_file)
            file_rows = pf.metadata.num_rows

            with tqdm(total=file_rows, desc=parquet_file.name, leave=False) as pbar:
                for batch in pf.iter_batches(
                    batch_size=batch_size,
                    columns=[dataset["code_col"]],
                ):
                    codes = batch.to_pydict()[dataset["code_col"]]

                    for code_value in codes:
                        current_source_idx = source_idx
                        source_idx += 1

                        code_text = normalize_text(code_value)
                        total_seen += 1

                        if not keep_by_dataset_sample(
                            source=source,
                            source_idx=current_source_idx,
                            sample_percent=sample_percent,
                        ):
                            sampled_out += 1
                            continue

                        if is_empty_code(code_text):
                            empty_code += 1
                        elif token_count(code_text) <= max_reference_tokens:
                            valid_count += 1
                        else:
                            too_long += 1

                    pbar.update(len(codes))

            gc.collect()

    print("\nFiltering summary:")
    print(f"Total rows seen: {total_seen}")
    print(f"Sampled out: {sampled_out}")
    print(f"Valid rows after sampling: {valid_count}")
    print(f"Filtered too long: {too_long}")
    print(f"Filtered empty code: {empty_code}")

    return valid_count, total_seen, too_long, empty_code, sampled_out


# ============================================================
# SPLIT
# ============================================================

def create_split_labels(valid_count: int) -> np.ndarray:
    print("\nSplitting valid items...")

    rng = np.random.default_rng(seed)

    n_code_corpus = round(valid_count * p_code_corpus / 100)
    n_val = round(valid_count * p_val / 100)
    n_train = valid_count - n_code_corpus - n_val

    # 0 = train, 1 = val, 2 = code_corpus
    split_labels = np.zeros(valid_count, dtype=np.uint8)

    indices = rng.permutation(valid_count)

    split_labels[indices[:n_code_corpus]] = 2
    split_labels[indices[n_code_corpus:n_code_corpus + n_val]] = 1

    del indices
    gc.collect()

    print(f"Target train rows: {n_train}")
    print(f"Target val rows: {n_val}")
    print(f"Target code corpus rows: {n_code_corpus}")

    return split_labels


# ============================================================
# PASS 2: WRITE FILES AND MANIFESTS
# ============================================================

def make_manifest_row(stem: str) -> dict[str, str]:
    return {
        "code_path_with_text": f"/data/references_with_text/{stem}.txt",
        "code_path_without_text": f"/data/references_without_text/{stem}.txt",
        "image_path_with_text": f"/data/images_with_text/{stem}.png",
        "image_path_without_text": f"/data/images_without_text/{stem}.png",
        "vlm_description_path": f"/data/vlm_descriptions/{stem}.txt",

        # Input ist mit Text.
        "input_image": f"/images_with_text/{stem}.png",
        "input_image_format_for_vlm": f"file:///images_with_text/{stem}.png",

        # Target / Ground Truth ist ohne Text.
        "reference_image": f"/images_without_text/{stem}.png",
        "reference_code": f"/references_without_text/{stem}.txt",

        "vlm_description": f"file:///vlm_descriptions/{stem}.txt",
    }


def write_item(stem: str, code_text: str, description_value) -> bool:
    code_without_text = replace_all_nodes(
        code_text,
        replacement=node_replacement_without_text,
    )

    image_with_text_path = image_with_text_folder / f"{stem}.png"
    image_without_text_path = image_without_text_folder / f"{stem}.png"

    with_text_ok = render_or_handle_failure(
        tex_code=code_text,
        output_path=image_with_text_path,
        stem=stem,
        variant="with_text",
    )

    without_text_ok = render_or_handle_failure(
        tex_code=code_without_text,
        output_path=image_without_text_path,
        stem=stem,
        variant="without_text",
    )

    if not (with_text_ok and without_text_ok):
        return False

    (code_with_text_folder / f"{stem}.txt").write_text(
        code_text,
        encoding="utf-8",
    )

    (code_without_text_folder / f"{stem}.txt").write_text(
        code_without_text,
        encoding="utf-8",
    )

    (description_folder / f"{stem}.txt").write_text(
        normalize_text(description_value),
        encoding="utf-8",
    )

    return True


def write_files_and_manifests(
    split_labels: np.ndarray,
) -> tuple[int, int, int, int, int]:
    train_manifest_path = output_dir / "manifest_train.csv"
    val_manifest_path = output_dir / "manifest_val.csv"

    written_train = 0
    written_val = 0
    written_code_corpus = 0
    valid_idx = 0
    render_failed = 0

    print("\nSecond pass: writing files, rendering images and writing manifests...")

    with train_manifest_path.open("w", newline="", encoding="utf-8") as train_file, \
         val_manifest_path.open("w", newline="", encoding="utf-8") as val_file:

        train_writer = csv.DictWriter(train_file, fieldnames=manifest_columns)
        val_writer = csv.DictWriter(val_file, fieldnames=manifest_columns)

        train_writer.writeheader()
        val_writer.writeheader()

        for dataset in datasets:
            source = dataset["source"]
            sample_percent = float(dataset.get("sample_percent", 100.0))
            source_idx = 0
            parquet_files = sorted(dataset["path"].glob("*.parquet"))

            print(f"\nSource: {source}")
            print(f"Sample percent: {sample_percent}%")

            for parquet_file in tqdm(parquet_files, desc=f"Writing {source}"):
                pf = pq.ParquetFile(parquet_file)

                columns = [dataset["code_col"]]

                if dataset["description_col"] is not None:
                    columns.append(dataset["description_col"])

                with tqdm(
                    total=pf.metadata.num_rows,
                    desc=parquet_file.name,
                    leave=False,
                ) as pbar:
                    for batch in pf.iter_batches(
                        batch_size=batch_size,
                        columns=columns,
                    ):
                        data = batch.to_pydict()

                        codes = data[dataset["code_col"]]

                        if dataset["description_col"] is None:
                            descriptions = [""] * len(codes)
                        else:
                            descriptions = data[dataset["description_col"]]

                        for description_value, code_value in zip(descriptions, codes):
                            current_source_idx = source_idx
                            stem = f"{source}_{current_source_idx:08d}"
                            source_idx += 1

                            code_text = normalize_text(code_value)

                            if not keep_by_dataset_sample(
                                source=source,
                                source_idx=current_source_idx,
                                sample_percent=sample_percent,
                            ):
                                continue

                            if is_empty_code(code_text):
                                continue

                            if token_count(code_text) > max_reference_tokens:
                                continue

                            split = split_labels[valid_idx]
                            valid_idx += 1

                            if split == 2:
                                code_corpus_path = code_corpus_folder / f"{stem}.txt"
                                code_corpus_path.write_text(
                                    code_text,
                                    encoding="utf-8",
                                )
                                written_code_corpus += 1
                                continue

                            if not write_item(stem, code_text, description_value):
                                render_failed += 1
                                continue

                            row = make_manifest_row(stem)

                            if split == 1:
                                val_writer.writerow(row)
                                written_val += 1
                            else:
                                train_writer.writerow(row)
                                written_train += 1

                        pbar.update(batch.num_rows)

                gc.collect()

    return (
        written_train,
        written_val,
        written_code_corpus,
        valid_idx,
        render_failed,
    )


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    setup_output_folders()
    print_config()

    valid_count, total_seen, too_long, empty_code, sampled_out = count_valid_items()

    split_labels = create_split_labels(valid_count)

    (
        written_train,
        written_val,
        written_code_corpus,
        valid_idx,
        render_failed,
    ) = write_files_and_manifests(split_labels)

    if valid_idx != valid_count:
        raise RuntimeError(
            f"Valid count mismatch: first pass found {valid_count}, "
            f"second pass wrote/processed {valid_idx}"
        )

    if not skip_render_failures and render_failed != 0:
        raise RuntimeError(f"Unexpected render failures: {render_failed}")

    print("\nDone.")
    print(f"Total rows seen: {total_seen}")
    print(f"Sampled out: {sampled_out}")
    print(f"Valid rows after sampling: {valid_count}")
    print(f"Filtered too long: {too_long}")
    print(f"Filtered empty code: {empty_code}")

    print()
    print(f"Train rows: {written_train}")
    print(f"Val rows: {written_val}")
    print(f"Code corpus rows: {written_code_corpus}")
    print(f"Render failed rows: {render_failed}")

    print()
    print(f"Train manifest: {output_dir / 'manifest_train.csv'}")
    print(f"Val manifest: {output_dir / 'manifest_val.csv'}")
    print(f"Code corpus folder: {code_corpus_folder}")
    print(f"Output folder: {output_dir}")


if __name__ == "__main__":
    main()

Output folder: /home/jonas/Datasets/TikZ/train-big
Batch size: 512
Max reference tokens: 8192
Validation split: 0.5%
Code corpus split: 5%
Skip render failures: False
Node replacement without text: 'Text'

Dataset sampling:
- datikz_v4: 100.0%
- geotikz_bridge_base: 50.0%

First pass: filtering by dataset sample and token length...

Source: datikz_v4
Parquet files: 86
Sample percent: 100.0%


Filtering datikz_v4: 100%|██████████| 86/86 [01:50<00:00,  1.29s/it]



Source: geotikz_bridge_base
Parquet files: 4
Sample percent: 50.0%


Filtering geotikz_bridge_base: 100%|██████████| 4/4 [04:43<00:00, 70.92s/it]



Filtering summary:
Total rows seen: 2881321
Sampled out: 1226741
Valid rows after sampling: 1645602
Filtered too long: 8978
Filtered empty code: 0

Splitting valid items...
Target train rows: 1555094
Target val rows: 8228
Target code corpus rows: 82280

Second pass: writing files, rendering images and writing manifests...

Source: datikz_v4
Sample percent: 100.0%


Writing datikz_v4:   0%|          | 0/86 [00:19<?, ?it/s]


TikzRenderError: Render failed:
pgfcorearrows.code.te
x)
(/usr/share/texlive/texmf-dist/tex/generic/pgf/basiclayer/pgfcoreshade.code.tex
)
(/usr/share/texlive/texmf-dist/tex/generic/pgf/basiclayer/pgfcoreimage.code.tex
)
(/usr/share/texlive/texmf-dist/tex/generic/pgf/basiclayer/pgfcoreexternal.code.
tex)
(/usr/share/texlive/texmf-dist/tex/generic/pgf/basiclayer/pgfcorelayers.code.te
x)
(/usr/share/texlive/texmf-dist/tex/generic/pgf/basiclayer/pgfcoretransparency.c
ode.tex)
(/usr/share/texlive/texmf-dist/tex/generic/pgf/basiclayer/pgfcorepatterns.code.
tex)
(/usr/share/texlive/texmf-dist/tex/generic/pgf/basiclayer/pgfcorerdf.code.tex))
)
(/usr/share/texlive/texmf-dist/tex/generic/pgf/modules/pgfmoduleshapes.code.tex
) (/usr/share/texlive/texmf-dist/tex/generic/pgf/modules/pgfmoduleplot.code.tex
)
(/usr/share/texlive/texmf-dist/tex/latex/pgf/compatibility/pgfcomp-version-0-65
.sty)
(/usr/share/texlive/texmf-dist/tex/latex/pgf/compatibility/pgfcomp-version-1-18
.sty)) (/usr/share/texlive/